## Sales Development Representative

In [1]:
# Import libraries
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content

In [2]:
# The usual starting point
load_dotenv(override=True)

True

In [3]:
# Step1: Agent workflow (simple workflow between three agents)
instructions1 = "You are a sales agent working for EthicsAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for EthicsAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for EthicsAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [4]:
# Make agents  with name, instructions, and model
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [5]:
# Convert agents to tools
description = "Generate a cold sales email to a potential customer for EthicsAI"

tool1 = sales_agent1.as_tool(tool_name="professional_sales_email_generator", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="engaging_sales_email_generator", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="busy_sales_email_generator", tool_description=description)

tools = [tool1, tool2, tool3]

In [6]:
# Create agents to write subject and convert email body to HTML
subject_instructions = "You can write a subject for a cold sales email. \
    You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
    You are given a text email body which might have some mardown \
        and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email Subject Writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML Email Body Converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [7]:
# Create a function tool to send out the email 
# (mocked out for now, but you could easily implement this with SendGrid or another email provider)
@function_tool
def send_html_email(subject:str, html_body:str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to all sales prospects"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("iwanttotestanapp@gmail.com")
    to_email = To("ctrlplusstyle@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [8]:
# List of tools for the agent to use in the workflow
em_tools = [subject_tool, html_tool, send_html_email]

In [9]:
# Create the agent that will use the above tools to complete the workflow of writing 
# and sending a cold sales email (Which is the Sales Maneger Agent)
instructions = "You are an email formatter and Writer. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the hmtl_converter tool to convert the body to html. \
Finally, you use the send_html_email tool to send the email with the subject and html body."

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=em_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it out"
)

In [10]:
# List of tools for the agent to use in the workflow
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [11]:
# Create the Sales Manager agent that will generate three different email drafts using the three sales agents
# Select the best one, and hand it off to the Email Manager to send out.
sales_manager_instructions = """
You are a Sales Manager at EthicsAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini"
)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated Sales Development Representative"):
    result = await Runner.run(sales_manager, input=message)


print(result.final_output)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


The cold sales email has been successfully sent! If you need any further assistance or have other emails to draft, feel free to let me know.


### Implementing Guardrails & Structured Outputs for Robust AI Agent Systems

In [17]:
from pydantic import BaseModel


# Define the output schema for the name check tool
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

# Create the guardrail agent that checks if the user is including someone's name in their request. 
guardrail_agent = Agent(
    name="Name check",
    instructions="Check if the user is including someone's name in what they want to you to do",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

In [26]:
# Create the guardrail function that uses the guardrail agent to check if the user is including someone's 
# name in their request, and if so, trigger the tripwire and return the name that was included.
from agents import input_guardrail, GuardrailFunctionOutput

@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, input=message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(
        output_info={"found_name": result.final_output}, 
        tripwire_triggered=is_name_in_message
    )

In [27]:
# Now we can add this guardrail to the Sales Manager agent to ensure that it does not process any requests 
# that include someone's name.
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
)

In [29]:
# Now if we try to run the careful_sales_manager with a message that includes a name, it should trigger the guardrail and not process the request.
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, input=message)

print(result.final_output)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

- Check out the trace

In [30]:
# And if we run it with a message that does not include a name, it should process the request as normal.
message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, input=message)

print(result.final_output)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function c

The cold sales email has been successfully sent to the CEO with the subject "Streamline Your SOC2 Compliance Journey with EthicsAI." If you need any further assistance or have additional tasks, feel free to ask!
